# FODO cell with finite dipoles

This notebook reproduces the calculations in the [MSU Accelerator Physics FODO-cell notes](https://msu-beam-dynamics.github.io/AP_enotes/transverse_BD/examples/FODO_cell.html) with one concrete TrackPad lattice. It covers transfer matrices, stability, periodic Twiss functions, dispersion, natural chromaticity, the dispersion invariant $\mathcal H$, the endpoint radiation-integral estimate, and phase-advance optimization.

The analytical reference uses thin quadrupoles and small-angle dipoles. The tracked cell uses finite-length `Quadrupole` and `SBend` elements, so small differences are physical approximation errors rather than numerical failures.

## Setup and conventions

The cell is a 3 GeV electron lattice with a target horizontal phase advance of $90^\circ$. Each half cell has length $L_1=5$ m and contains one sector bend of $0.03$ rad. The quadrupoles are 2 cm long.

TrackPad stores $\delta_E=(E-E_0)/(P_0c)$, while the notes use $\delta_P=(P-P_0)/P_0$. At first order, $\delta_E=\beta_0\delta_P$, so the notebook applies the corresponding $\beta_0$ conversion to dispersion and chromaticity.

In [ ]:
using Pkg

repo_root = let
    candidates = (pwd(), dirname(pwd()))
    index = findfirst(path -> isfile(joinpath(path, "Project.toml")) &&
        isfile(joinpath(path, "examples", "advanced", "fodo_cell_derivations.jl")), candidates)
    isnothing(index) && error("Start Jupyter from the TrackPad root or examples directory")
    candidates[index]
end
Pkg.activate(joinpath(repo_root, "examples"))
Pkg.resolve()
Pkg.instantiate()

using LinearAlgebra
using CairoMakie
using TrackPad
CairoMakie.activate!(type = "svg")

if !isdefined(Main, :FODOCellWithDipolesExample)
    Base.include(Main, joinpath(repo_root, "examples", "advanced", "fodo_cell_derivations.jl"))
end
using .FODOCellWithDipolesExample

result = run_example(; check = true, verbose = false);

In [ ]:
parameters = (
    kinetic_energy_GeV = result.beam.energy / 1e9,
    beta0 = result.beam.beta,
    half_cell_length_m = 5.0,
    quadrupole_length_m = 0.02,
    bend_angle_rad = 0.03,
    thin_lens_focal_length_m = 5 / (2sin(pi / 4)),
)
lattice_elements = [
    (name = element.name, type = nameof(typeof(element)), length_m = element.L)
    for element in result.lattices.mid_qf
]
(; parameters, lattice_elements)

## 1. Transfer matrices and stability

At the midpoint of QF, the thin-lens horizontal map is

$$M_{FODO}=\begin{pmatrix}1-L_1^2/(2f^2) & L_1(L_1+2f)/f \\ L_1(L_1-2f)/(4f^3) & 1-L_1^2/(2f^2)\end{pmatrix}.$$

Its determinant is one and $\operatorname{tr}M=2-L_1^2/f^2$. Stability requires $L_1<2f$, and $\sin(\Phi/2)=L_1/(2f)$. Cyclically changing the starting point preserves the trace.

**Convention correction:** Equations 9.4 and 9.9 in the source notes are labeled “after QF,” but their matrix is the after-QD cyclic map under the focusing signs used in Equation 9.1. Both the published matrix and the physically after-QF map are shown below.

In [ ]:
matrix_comparison = (
    thin_mid_QF = result.thin.mid_qf,
    tracked_mid_QF = result.maps.mid_qf[1:2, 1:2],
    thin_mid_QD = result.thin.mid_qd,
    tracked_mid_QD = result.maps.mid_qd[1:2, 1:2],
    published_Eq_9_4 = result.thin.note_after_qf,
    tracked_at_matching_after_QD_boundary = result.maps.after_qd[1:2, 1:2],
    physical_after_QF = result.maps.after_qf[1:2, 1:2],
)

In [ ]:
matrix_checks = (
    determinant_mid_QF = det(result.maps.mid_qf[1:2, 1:2]),
    determinant_mid_QD = det(result.maps.mid_qd[1:2, 1:2]),
    trace_mid_QF = tr(result.maps.mid_qf[1:2, 1:2]),
    trace_mid_QD = tr(result.maps.mid_qd[1:2, 1:2]),
    target_phase_deg = 90.0,
    tracked_phase_deg = rad2deg(result.optics.mid_qf.mu),
    relative_matrix_errors = (
        mid_QF = result.errors.matrix_mid_qf,
        mid_QD = result.errors.matrix_mid_qd,
        after_QF = result.errors.matrix_after_qf,
        published_Eq_9_4 = result.errors.matrix_note_after_qf,
    ),
)

## 2. Periodic Twiss functions and dispersion

For a stable uncoupled map, $\beta=M_{12}/\sin\Phi$ and $\alpha=(M_{11}-M_{22})/(2\sin\Phi)$. Symmetry gives $\alpha=0$ at both quadrupole centers. The thin-lens extrema are

$$\beta_{F/D}=\frac{2L_1[1\pm\sin(\Phi/2)]}{\sin\Phi}.$$

The periodic dispersion is obtained from $(I-A)(D,D')^T=b$, where $A$ is the transverse block and $b$ is the map's energy-coordinate column.

In [ ]:
periodic_optics = (
    beta_QF_m = result.optics.mid_qf.beta,
    beta_QD_m = result.optics.mid_qd.beta,
    alpha_QF = result.optics.mid_qf.alpha,
    alpha_QD = result.optics.mid_qd.alpha,
    dispersion_QF_m = result.dispersion.mid_qf,
    dispersion_QD_m = result.dispersion.mid_qd,
    thin_beta_QF_m = result.analytic.beta_f,
    thin_beta_QD_m = result.analytic.beta_d,
    thin_dispersion_QF_m = result.analytic.dispersion_f,
    thin_dispersion_QD_m = result.analytic.dispersion_d,
)

## Optics functions through the cell

For a smooth longitudinal plot, the finite magnets are sliced into short tracking elements without changing their integrated strengths or bend angles. `twissline` supplies both periodic beta functions. The periodic dispersion is propagated through the same segment Jacobians and converted from TrackPad's energy coordinate to the momentum convention used in the notes.

In [ ]:
set_theme!(Theme(
    fontsize = 15,
    backgroundcolor = :white,
    Axis = (
        backgroundcolor = :white,
        xgridcolor = (:gray, 0.18),
        ygridcolor = (:gray, 0.18),
    ),
))

In [ ]:
sampled_optics = sample_fodo_optics()

optics_figure = Figure(size = (920, 520))
beta_axis = Axis(
    optics_figure[1, 1],
    title = "Periodic optics through the finite FODO cell",
    xlabel = "s [m]", ylabel = "β function [m]",
)
dispersion_axis = Axis(
    optics_figure[1, 1],
    yaxisposition = :right, ylabel = "horizontal dispersion Dₓ [m]",
    backgroundcolor = :transparent,
)
linkxaxes!(beta_axis, dispersion_axis)
hidexdecorations!(dispersion_axis)
hidespines!(dispersion_axis, :l, :b, :t)

vspan!(beta_axis, 0.01, 4.99; color = (:orange, 0.06))
vspan!(beta_axis, 5.01, 9.99; color = (:orange, 0.06))
vlines!(beta_axis, [0.0, 5.0, 10.0]; color = (:gray, 0.45), linestyle = :dash)
lines!(beta_axis, sampled_optics.s, sampled_optics.betax; color = "#005f73", linewidth = 3, label = "βₓ")
lines!(beta_axis, sampled_optics.s, sampled_optics.betay; color = "#ca6702", linewidth = 3, label = "βᵧ")
lines!(dispersion_axis, sampled_optics.s, sampled_optics.dispersion; color = "#9b2226", linewidth = 3, label = "Dₓ")
text!(beta_axis, [0.12, 5.0], fill(17.6, 2); text = ["QF", "QD"], align = (:center, :center), color = :gray35)
axislegend(beta_axis; position = :rt)
axislegend(dispersion_axis; position = :rb)
xlims!(beta_axis, 0, 10)
optics_figure

## Optics scaling with phase advance

The next figure reproduces the normalized thin-lens beta and dispersion scan from the notes.

In [ ]:
phase_deg = rad2deg.(result.scan.phase)
phase_figure = Figure(size = (920, 520))
beta_scan_axis = Axis(
    phase_figure[1, 1], title = "Normalized FODO optics versus phase advance",
    xlabel = "phase advance [deg]", ylabel = "β / L₁",
)
dispersion_scan_axis = Axis(
    phase_figure[1, 1], yaxisposition = :right,
    ylabel = "D / (L₁ θ)", backgroundcolor = :transparent,
)
linkxaxes!(beta_scan_axis, dispersion_scan_axis)
hidexdecorations!(dispersion_scan_axis)
hidespines!(dispersion_scan_axis, :l, :b, :t)
lines!(beta_scan_axis, phase_deg, result.scan.beta_max; color = "#005f73", linewidth = 3, label = "beta max")
lines!(beta_scan_axis, phase_deg, result.scan.beta_min; color = "#0a9396", linewidth = 3, label = "beta min")
lines!(dispersion_scan_axis, phase_deg, result.scan.dispersion_max; color = "#ca6702", linewidth = 3, label = "D max")
lines!(dispersion_scan_axis, phase_deg, result.scan.dispersion_min; color = "#bb3e03", linewidth = 3, label = "D min")
xlims!(beta_scan_axis, 0, 180); ylims!(beta_scan_axis, 0, 10); ylims!(dispersion_scan_axis, 0, 10)
axislegend(beta_scan_axis; position = :lt)
axislegend(dispersion_scan_axis; position = :rt)
phase_figure

## 3. Natural chromaticity and the $\mathcal H$ function

For the thin-lens cell,

$$\xi_x=-\frac{\beta_{max}-\beta_{min}}{4\pi f}=-\frac{\tan(\Phi/2)}{\pi}.$$

At the symmetric quadrupole centers, $D'=\alpha=0$, so $\mathcal H=D^2/\beta$. Replacing the radiation integral through each dipole with the average endpoint value gives $\rho I_5/I_2\simeq(\mathcal H_F+\mathcal H_D)/2$.

In [ ]:
radiation_and_chromaticity = (
    TrackPad_chromaticity_dQ_d_deltaP = result.chrom_momentum,
    thin_lens_chromaticity = result.analytic.chromaticity,
    H_QF_m = result.h.mid_qf,
    H_QD_m = result.h.mid_qd,
    rho_m = result.radiation.rho,
    I2_per_m = result.radiation.i2,
    I5_per_m = result.radiation.i5_endpoint,
    rho_I5_over_I2_m = result.radiation.rho_i5_over_i2,
)

In [ ]:
h_figure = Figure(size = (920, 520))
h_axis = Axis(
    h_figure[1, 1], title = "Dispersion invariant and specific chromaticity",
    xlabel = "phase advance [deg]", ylabel = "ℋ / (L₁ θ²)",
)
chrom_axis = Axis(
    h_figure[1, 1], yaxisposition = :right,
    ylabel = "ξₓ / νₓ", backgroundcolor = :transparent,
)
linkxaxes!(h_axis, chrom_axis)
hidexdecorations!(chrom_axis)
hidespines!(chrom_axis, :l, :b, :t)
lines!(h_axis, phase_deg, result.scan.h_average; color = "#005f73", linewidth = 3, label = "average ℋ")
lines!(h_axis, phase_deg, result.scan.h_f; color = ("#0a9396", 0.65), linewidth = 2, label = "ℋ at QF")
lines!(h_axis, phase_deg, result.scan.h_d; color = ("#ca6702", 0.65), linewidth = 2, label = "ℋ at QD")
lines!(chrom_axis, phase_deg, result.scan.specific_chromaticity; color = "#9b2226", linewidth = 3, label = "ξₓ / νₓ")
vlines!(h_axis, [result.optimum.grid_degrees]; color = (:gray, 0.7), linestyle = :dot)
vlines!(h_axis, [result.optimum.degrees]; color = (:black, 0.75), linestyle = :dash)
scatter!(h_axis, [result.optimum.degrees], [result.optimum.normalized_h]; color = :black, markersize = 10)
xlims!(h_axis, 0, 180); ylims!(h_axis, 0, 10); ylims!(chrom_axis, -8, 0)
axislegend(h_axis; position = :lt)
axislegend(chrom_axis; position = :rt)
h_figure

## 4. Optimum phase advance

The notes report $138.393853^\circ$, which is the minimum sampled by their 100-point plotting grid. Analytically minimizing the same endpoint estimate gives $137.959019^\circ$ and a slightly lower normalized $\langle\mathcal H\rangle$.

In [ ]:
optimization = (
    published_grid_phase_deg = result.optimum.grid_degrees,
    published_grid_normalized_H = result.optimum.grid_normalized_h,
    continuous_phase_deg = result.optimum.degrees,
    continuous_normalized_H = result.optimum.normalized_h,
    all_example_checks_passed = true,
)